---
title: "Machine Learning: Probabilistic Models and Bayesian Learning"
lang: en
format:
  html:
    toc: true
    toc-depth: 5
    theme: cosmo
jupyter: python
---


<div class="blog-language-switch" role="group" aria-label="Article language">
<span aria-current="page">English</span>
<a href="../zh-CN/Machine-Learning/10-probabilistic-bayesian-learning.html" lang="zh-CN" hreflang="zh-CN">中文</a>
</div>

[Back to Machine Learning guideline](Machine Learning.html)


## **Probabilistic Models and Bayesian Learning**

A point predictor returns one answer: a class, a number, or a ranking score. A **probabilistic model** returns a distribution over possible answers and states the assumptions used to construct that distribution. This difference matters whenever two predictions with the same most likely label should not be treated as equally certain, or whenever the cost of a mistake depends on which mistake is made.

Probabilistic learning separates three related objects:

1. A **sampling model** describes how observations could be generated for fixed parameters, such as $p(y\mid x,\theta)$.
2. A **parameter distribution** describes what is known about $\theta$. In Bayesian learning this is updated from a prior $p(\theta)$ to a posterior $p(\theta\mid\mathcal D)$.
3. A **predictive distribution** averages the possible outcomes implied by the model and the available parameter knowledge.

The adjective *probabilistic* is broader than *Bayesian*. Logistic regression, maximum-likelihood Gaussian naive Bayes, and a calibrated random forest all produce probabilities without necessarily placing a posterior distribution over parameters. Bayesian learning adds a probability model for unknown quantities and propagates their uncertainty into prediction. It is therefore important to distinguish a probability-valued output from full posterior inference.

<div class="diagram-scroll">

![A probabilistic learning workflow from observations and model assumptions to a posterior predictive distribution and a loss-aware action.](assets/probabilistic-learning-map.svg){fig-alt="A probabilistic learning workflow from observations and model assumptions to a posterior predictive distribution and a loss-aware action."}

</div>

The workflow has two layers that are often conflated. **Inference** uses data to obtain a distribution over unknowns or outcomes. **Decision making** combines that distribution with a loss or utility function to select an action. A 35% event probability is a statement about uncertainty; whether 35% is high enough to intervene depends on the relative consequences of intervention and inaction.

| Question | Mathematical object | Typical output |
|---|---|---|
| How could the data arise? | Likelihood $p(\mathcal D\mid\theta)$ | Fit of parameters to observations |
| What is known about parameters? | Posterior $p(\theta\mid\mathcal D)$ | Credible regions and parameter draws |
| What could happen next? | Predictive $p(y_*\mid x_*,\mathcal D)$ | Probabilities, means, quantiles, intervals |
| What should be done? | Posterior risk $R(a\mid x_*,\mathcal D)$ | A minimum-risk action |
| Can the uncertainty be trusted? | Calibration, coverage, diagnostics | Reliability evidence and limitations |

This chapter develops that chain in increasing order of difficulty. Naive Bayes and Gaussian discriminant analysis show how explicit distributional assumptions create classifiers. Conjugate models show exact Bayesian updating. Bayesian regression and Gaussian processes propagate parameter or function uncertainty. Approximate inference handles posteriors for which exact integration is unavailable. The final section returns to the practical question: whether a model's probabilities and intervals are reliable enough to support a decision.


### **Bayesian Decision Theory**

Bayesian decision theory connects uncertain beliefs to actions. It does not say that every system must use a Bayesian parameter prior. It says that once an appropriate predictive distribution is available, the rational action under a stated loss function is the one with the smallest expected loss. This exposes assumptions that a fixed threshold or raw accuracy score would otherwise hide.

#### **Posterior Prediction**

For training data $\mathcal D$, a new input $x_*$, and model parameters $\theta$, Bayesian prediction integrates over the posterior rather than pretending that one fitted parameter value is certainly correct:

$$
p(y_*\mid x_*,\mathcal D)
=\int p(y_*\mid x_*,\theta)\,p(\theta\mid\mathcal D)\,d\theta.
$$

The two factors play different roles. The first, $p(y_*\mid x_*,\theta)$, represents variability in the outcome for a fixed model. The second, $p(\theta\mid\mathcal D)$, represents uncertainty about the model after observing finite data. Their integral is a weighted average of predictions from all plausible parameter values.

A **plug-in prediction** replaces the integral with $p(y_*\mid x_*,\hat\theta)$, where $\hat\theta$ might be the maximum-likelihood estimate, MAP estimate, or posterior mean. Plug-in prediction is often cheap and accurate when the posterior is tightly concentrated and the prediction is nearly linear in $\theta$. It can be overconfident with little data, weak identification, extrapolation, or nonlinear transformations because $p(y_*\mid x_*,E[\theta])$ generally differs from $E[p(y_*\mid x_*,\theta)]$.

For a Bernoulli event with success probability $\theta$ and a $\operatorname{Beta}(\alpha,\beta)$ prior, observing $s$ successes and $f$ failures gives

$$
\theta\mid\mathcal D\sim\operatorname{Beta}(\alpha+s,\beta+f),
\qquad
P(y_*=1\mid\mathcal D)=\frac{\alpha+s}{\alpha+\beta+s+f}.
$$

The final expression is the posterior predictive probability. It is also the posterior mean of $\theta$, but that equality is specific to this Bernoulli prediction. The posterior distribution still contains information about how uncertain the probability itself is.

<details>
<summary><strong>Python example: analytic and Monte Carlo posterior prediction</strong></summary>

```python
import numpy as np
from scipy.stats import beta

# Prior knowledge and observed Bernoulli outcomes.
alpha_prior, beta_prior = 2.0, 2.0
successes, failures = 7, 3
alpha_post = alpha_prior + successes
beta_post = beta_prior + failures

# Closed-form posterior predictive probability for the next outcome.
analytic_predictive = alpha_post / (alpha_post + beta_post)

# Monte Carlo integration: average p(y*=1 | theta)=theta over posterior draws.
rng = np.random.default_rng(7)
theta_draws = rng.beta(alpha_post, beta_post, size=100_000)
mc_predictive = theta_draws.mean()
credible_interval = beta.ppf([0.025, 0.975], alpha_post, beta_post)

print(f"posterior: Beta({alpha_post:.0f}, {beta_post:.0f})")
print(f"analytic predictive probability: {analytic_predictive:.4f}")
print(f"Monte Carlo predictive probability: {mc_predictive:.4f}")
print(f"95% credible interval for theta: {credible_interval.round(3)}")
```

</details>

The posterior interval above concerns the unknown event rate $\theta$. It is not a 95% prediction interval for one future binary outcome, because that outcome can only be 0 or 1. Parameter uncertainty, outcome uncertainty, and uncertainty in an estimated numerical summary should always be named separately.

#### **Loss, Risk, and Bayes-Optimal Decisions**

Suppose an action $a$ is chosen before the true state $y$ is observed, and $L(a,y)$ is the loss incurred. The **posterior risk** is

$$
R(a\mid x,\mathcal D)
=E[L(a,Y)\mid x,\mathcal D]
=\sum_y L(a,y)p(y\mid x,\mathcal D)
$$

for discrete outcomes, with the sum replaced by an integral for continuous outcomes. The **Bayes action** is

$$
a^*(x)=\arg\min_a R(a\mid x,\mathcal D).
$$

Under symmetric zero-one loss, a correct classification costs 0 and any incorrect classification costs 1. Minimizing risk then selects the most probable class, so the Bayes action equals the MAP class. This familiar rule is therefore a special case, not a universal principle.

For binary prediction let $p=P(Y=1\mid x,\mathcal D)$, let a false positive cost $C_{FP}$, and let a false negative cost $C_{FN}$. The risks are

$$
R(\hat y=1)=C_{FP}(1-p),
\qquad
R(\hat y=0)=C_{FN}p.
$$

Predict class 1 when

$$
p>\frac{C_{FP}}{C_{FP}+C_{FN}}.
$$

If false negatives cost five times as much as false positives, the threshold is $1/(1+5)\approx0.167$, not 0.5. The threshold follows from consequences and must be estimated on a validation distribution representative of deployment; it should not be tuned on the final test set.

<div class="diagram-scroll">

![Posterior probabilities and a loss matrix are combined into posterior risks before selecting the Bayes action.](assets/bayes-decision-risk.svg){fig-alt="Posterior probabilities and a loss matrix are combined into posterior risks before selecting the Bayes action."}

</div>

A system may also support **abstention** or referral. If rejecting a case has fixed cost $C_R$, rejection is optimal whenever $C_R$ is smaller than the risks of both automatic actions. This creates a principled human-review region instead of treating confidence as an informal warning.

<details>
<summary><strong>Python example: choose minimum-risk actions with asymmetric costs and abstention</strong></summary>

```python
import numpy as np

prob_positive = np.array([0.05, 0.20, 0.35, 0.60, 0.92])
false_positive_cost = 1.0
false_negative_cost = 5.0
reject_cost = 0.45

# Rows are cases; columns are actions [predict 0, predict 1, reject].
risk_predict_0 = false_negative_cost * prob_positive
risk_predict_1 = false_positive_cost * (1.0 - prob_positive)
risk_reject = np.full_like(prob_positive, reject_cost)
risks = np.column_stack([risk_predict_0, risk_predict_1, risk_reject])

labels = np.array(["predict 0", "predict 1", "refer"])
actions = labels[np.argmin(risks, axis=1)]
threshold_without_rejection = (
    false_positive_cost / (false_positive_cost + false_negative_cost)
)

print(f"cost-sensitive threshold without rejection: {threshold_without_rejection:.3f}")
for probability, case_risks, action in zip(prob_positive, risks, actions):
    print(f"p={probability:.2f} risks={case_risks.round(2)} -> {action}")
```

</details>

| Loss structure | Bayes-optimal output | What must be specified |
|---|---|---|
| Symmetric zero-one | Most probable class | Class posterior probabilities |
| Asymmetric binary costs | Cost-derived probability threshold | False-positive and false-negative costs |
| Squared error | Posterior predictive mean | Relative penalty grows quadratically |
| Absolute error | Posterior predictive median | Equal linear penalty on either side |
| Reject option | Predict or abstain by minimum risk | Cost of referral, delay, or no decision |

Bayesian decision theory is valuable even when stakeholders cannot provide exact monetary costs. Writing a plausible loss matrix and testing conclusions across a range of costs makes the operational trade-off visible. A model with slightly lower accuracy can be preferable if it assigns well-calibrated probabilities and therefore supports better decisions under the losses that actually matter.


### **Generative and Discriminative Modeling**

Probabilistic classifiers can be organized by which distribution they model. A **generative classifier** models how features and labels occur together. A **discriminative classifier** directly models the label conditional on the observed features, or learns a decision function from which a conditional probability can be obtained.

#### **Modeling Joint and Conditional Distributions**

A generative classifier factorizes the joint distribution as

$$
p(x,y)=p(y)p(x\mid y).
$$

At prediction time, Bayes' rule converts this into

$$
p(y=c\mid x)
=\frac{p(y=c)p(x\mid y=c)}{\sum_k p(y=k)p(x\mid y=k)}.
$$

The class prior $p(y=c)$ describes prevalence, while the class-conditional density $p(x\mid y=c)$ describes what observations from class $c$ look like. Naive Bayes, LDA, QDA, hidden Markov models, and many mixture models use this route. Because a joint model specifies a distribution for $x$, it can in principle generate synthetic observations, evaluate whether an input is plausible, incorporate unlabeled observations in some settings, and reason about missing features by marginalization.

A discriminative probabilistic model instead targets

$$
p(y\mid x;\theta)
$$

directly. Logistic regression, conditional random fields, and discriminatively trained neural classifiers are examples. A model such as a support vector machine is also discriminative, although its raw margin is not automatically a calibrated probability. By not spending capacity on $p(x)$, a discriminative model can focus on the boundary or conditional probability needed for prediction.

<div class="diagram-scroll">

![Generative models learn a joint distribution and use Bayes rule, whereas discriminative models directly learn a conditional distribution or decision score.](assets/generative-discriminative-map.svg){fig-alt="Generative models learn a joint distribution and use Bayes rule, whereas discriminative models directly learn a conditional distribution or decision score."}

</div>

The terms describe factorization, not model quality. A generative model can be highly flexible, and a discriminative model can be badly misspecified. Similarly, generating text from a conditional language model does not by itself make every component of the system a generative classifier in this statistical sense.

#### **Strengths and Trade-Offs**

Generative assumptions can reduce the amount of data required. If each class really is a Gaussian with independent features, estimating class means and variances is statistically efficient. A discriminative boundary may need more labeled examples to discover the same regularity. The advantage reverses when the assumed feature distribution is wrong: a direct conditional model avoids fitting irrelevant aspects of $x$ and may approach a better asymptotic decision boundary.

This produces a useful bias-variance trade-off:

- **Strong, approximately correct assumptions** can make a generative model learn quickly from small samples.
- **Weak or violated assumptions** can bias its probabilities even when classification accuracy remains acceptable.
- **Discriminative training** often wins with abundant labeled data, but it does not automatically solve calibration, extrapolation, or distribution shift.
- **Missing features** can be handled naturally only when the generative model defines the required marginals and the missingness mechanism is defensible.

<details>
<summary><strong>Python example: compare generative and discriminative learning as sample size grows</strong></summary>

```python
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import log_loss
from sklearn.naive_bayes import GaussianNB

rng = np.random.default_rng(12)

def sample_gaussian_classes(n, rng):
    # The GaussianNB assumptions are deliberately correct in this experiment.
    y = rng.integers(0, 2, size=n)
    means = np.where(y[:, None] == 1, 0.65, -0.65)
    X = rng.normal(loc=means, scale=1.0, size=(n, 8))
    return X, y

X_test, y_test = sample_gaussian_classes(20_000, rng)

for n_train in [20, 50, 200, 1_000]:
    scores = {"GaussianNB": [], "LogisticRegression": []}
    for repeat in range(20):
        X_train, y_train = sample_gaussian_classes(n_train, rng)
        models = {
            "GaussianNB": GaussianNB(),
            "LogisticRegression": LogisticRegression(max_iter=2_000),
        }
        for name, model in models.items():
            model.fit(X_train, y_train)
            probability = model.predict_proba(X_test)[:, 1]
            scores[name].append(log_loss(y_test, probability))

    result = {name: np.mean(values) for name, values in scores.items()}
    print(f"n={n_train:4d}: " + ", ".join(
        f"{name} log-loss={value:.3f}" for name, value in result.items()
    ))
```

</details>

This experiment is intentionally favorable to Gaussian naive Bayes. It demonstrates what correct structural assumptions can buy; it does not establish a universal ranking. A responsible comparison should repeat the experiment under correlated, heavy-tailed, and shifted features, then evaluate both decisions and probability quality.

| Requirement | Generative route | Discriminative route |
|---|---|---|
| Primary target | $p(x,y)$ or $p(y)p(x\mid y)$ | $p(y\mid x)$ or a decision score |
| Typical benefit | Data efficiency and richer probabilistic queries | Focused predictive fit with fewer assumptions on $x$ |
| Main risk | Joint-model misspecification | Limited answers outside the trained conditional task |
| Missing or latent variables | Natural if the full model is credible | Usually requires separate handling |
| Synthetic observations | Defined by the model | Usually not defined |
| Calibration | Not guaranteed | Not guaranteed |

The practical question is therefore not “generative or discriminative?” in isolation. It is which assumptions are credible, which outputs are required, how much labeled data is available, and how errors will be judged in deployment.


### **Naive Bayes**

Naive Bayes is a family of generative classifiers built around one strong simplification: features are conditionally independent once the class is known. The assumption is rarely literally true, but it converts a difficult high-dimensional density estimate into many small one-dimensional estimates. The resulting model is fast, works naturally with sparse data, and is an important baseline for text and categorical problems.

For feature vector $x=(x_1,\ldots,x_d)$ and class $c$,

$$
p(x\mid y=c)=\prod_{j=1}^{d}p(x_j\mid y=c),
$$

so classification uses

$$
\hat y
=\arg\max_c p(y=c)\prod_{j=1}^{d}p(x_j\mid y=c).
$$

Products of many probabilities can underflow numerically. Implementations therefore compare log scores:

$$
g_c(x)=\log p(y=c)+\sum_{j=1}^{d}\log p(x_j\mid y=c).
$$

The normalizing denominator $p(x)$ is the same for every class and is omitted when only the winning class is needed. It must be restored, usually through a log-sum-exp normalization, when posterior probabilities are required.

<div class="diagram-scroll">

![A Naive Bayes graph in which the class is the common parent of conditionally independent feature variables.](assets/naive-bayes-factorization.svg){fig-alt="A Naive Bayes graph in which the class is the common parent of conditionally independent feature variables."}

</div>

#### **Gaussian, Multinomial, Bernoulli, and Categorical Models**

The word *naive* identifies the conditional-independence structure; the feature likelihood still has to match the data representation.

| Variant | Feature model | Suitable representation | Important assumption |
|---|---|---|---|
| Gaussian NB | $x_j\mid y=c\sim\mathcal N(\mu_{cj},\sigma_{cj}^2)$ | Continuous measurements | Each feature is class-conditionally Gaussian |
| Multinomial NB | Count vector with class-specific event probabilities | Token counts, event counts | Counts arise from repeated categorical events |
| Bernoulli NB | $x_j\in\{0,1\}$ | Token presence, binary indicators | Presence/absence is informative; repeats are ignored |
| Categorical NB | One categorical distribution per feature and class | Color, region, device type | Categories are discrete states, not magnitudes |

**Gaussian NB** estimates one mean and variance for every class-feature pair. It creates quadratic-looking contributions in individual dimensions, although its diagonal covariance assumption excludes within-class feature correlations. Scaling is not mathematically required because each feature receives its own variance, but transformations may improve the Gaussian approximation.

**Multinomial NB** is common in text classification. For class $c$, token $j$ has probability $\phi_{cj}$ and a document contributes its token counts $x_j$:

$$
\log p(x\mid y=c)=\text{constant}(x)+\sum_j x_j\log\phi_{cj}.
$$

Repeated occurrences therefore add repeated evidence. Document length affects the likelihood, so the model is about count-generating events rather than Euclidean distance between documents.

**Bernoulli NB** first binarizes each feature. It scores both present and absent events,

$$
\log p(x\mid y=c)
=\sum_j x_j\log\phi_{cj}+(1-x_j)\log(1-\phi_{cj}),
$$

which can be preferable when a word's presence matters but repetition adds little. **Categorical NB** instead treats each input column as a multi-valued state. Integer codes are labels, not ordered quantities.

<details>
<summary><strong>Python example: implement categorical Naive Bayes with an unknown category</strong></summary>

```python
from collections import Counter
import numpy as np

class CategoricalNaiveBayes:
    def __init__(self, alpha=1.0):
        self.alpha = float(alpha)

    def fit(self, X, y):
        X = np.asarray(X, dtype=object)
        y = np.asarray(y, dtype=object)
        self.classes_, class_counts = np.unique(y, return_counts=True)
        self.log_prior_ = {
            c: np.log(count / len(y))
            for c, count in zip(self.classes_, class_counts)
        }
        self.levels_ = [set(X[:, j]) | {"<UNK>"} for j in range(X.shape[1])]
        self.log_prob_ = {}

        for c in self.classes_:
            rows = X[y == c]
            self.log_prob_[c] = []
            for j, levels in enumerate(self.levels_):
                counts = Counter(rows[:, j])
                denominator = len(rows) + self.alpha * len(levels)
                table = {
                    level: np.log((counts[level] + self.alpha) / denominator)
                    for level in levels
                }
                self.log_prob_[c].append(table)
        return self

    def predict(self, X):
        X = np.asarray(X, dtype=object)
        predictions = []
        for row in X:
            scores = {}
            for c in self.classes_:
                score = self.log_prior_[c]
                for j, value in enumerate(row):
                    level = value if value in self.levels_[j] else "<UNK>"
                    score += self.log_prob_[c][j][level]
                scores[c] = score
            predictions.append(max(scores, key=scores.get))
        return np.asarray(predictions)

X = [
    ["sunny", "high"], ["sunny", "normal"], ["rain", "high"],
    ["rain", "normal"], ["overcast", "high"], ["overcast", "normal"],
]
y = ["stay", "go", "stay", "go", "go", "go"]

model = CategoricalNaiveBayes(alpha=1.0).fit(X, y)
print(model.predict([["sunny", "high"], ["snow", "normal"]]))
```

</details>

<details>
<summary><strong>Python example: classify a tiny text corpus with Multinomial Naive Bayes</strong></summary>

```python
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import make_pipeline

# English examples are kept because token identity is part of the experiment.
documents = [
    "team wins match", "player scores goal", "coach plans training",
    "market stocks rise", "investors sell shares", "company reports profit",
]
labels = ["sport", "sport", "sport", "finance", "finance", "finance"]

model = make_pipeline(
    CountVectorizer(ngram_range=(1, 2)),
    MultinomialNB(alpha=1.0),
)
model.fit(documents, labels)

examples = ["team scores goal", "company shares rise"]
probabilities = model.predict_proba(examples)
for text, prediction, probability in zip(
    examples, model.predict(examples), probabilities
):
    print(text, "->", prediction, dict(zip(model.classes_, probability.round(3))))
```

</details>

<details>
<summary><strong>Python example: contrast Bernoulli and Multinomial event models</strong></summary>

```python
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import BernoulliNB, MultinomialNB

train = [
    "win prize now", "win win cash prize", "claim cash now",
    "project meeting today", "meeting schedule project", "team meeting notes",
]
y = ["spam", "spam", "spam", "work", "work", "work"]
test = ["win win win meeting"]

vectorizer = CountVectorizer()
X = vectorizer.fit_transform(train)
X_test = vectorizer.transform(test)

models = {
    "Multinomial (repetitions count)": MultinomialNB(alpha=1.0),
    "Bernoulli (presence only)": BernoulliNB(alpha=1.0, binarize=0.0),
}
for name, model in models.items():
    model.fit(X, y)
    proba = dict(zip(model.classes_, model.predict_proba(X_test)[0].round(3)))
    print(name, "->", proba)
```

</details>

#### **Conditional Independence and Smoothing**

Conditional independence means

$$
X_j\perp X_k\mid Y
$$

for every feature pair in the model. It does **not** mean that features are marginally independent. Two words may co-occur frequently because both are associated with the same topic; this dependence can disappear or weaken after conditioning on the topic class. The assumption fails when features remain redundant inside each class, such as including a measurement and several near-duplicate transformations of it.

Naive Bayes can classify well despite dependence because the correct ranking of class scores is weaker than the requirement for a correct joint density. Its posterior probabilities are more fragile: repeated correlated evidence is multiplied several times, often producing excessive confidence.

For a multinomial event vocabulary of size $V$, additive smoothing estimates

$$
\hat\phi_{cj}
=\frac{N_{cj}+\alpha}{\sum_{k=1}^{V}N_{ck}+\alpha V},
$$

where $N_{cj}$ is the count of event $j$ in class $c$. With $\alpha=1$ this is Laplace smoothing. Smoothing prevents an unseen event from assigning the entire document zero probability under a class. It also shrinks rare estimates toward a more uniform distribution. The denominator must match the event model: categorical features have separate category tables per column, while multinomial text uses a shared event vocabulary.

<details>
<summary><strong>Python example: observe overconfidence from conditionally correlated features</strong></summary>

```python
import numpy as np
from sklearn.metrics import accuracy_score, brier_score_loss, log_loss
from sklearn.naive_bayes import GaussianNB
from sklearn.model_selection import train_test_split

rng = np.random.default_rng(21)
n = 8_000
y = rng.integers(0, 2, size=n)
latent = rng.normal(loc=np.where(y == 1, 0.8, -0.8), scale=1.0)

# The extra columns are near-duplicates, not independent new evidence.
X_single = latent[:, None]
X_redundant = np.column_stack([
    latent + rng.normal(scale=0.05, size=n) for _ in range(8)
])

indices = np.arange(n)
train, test = train_test_split(indices, test_size=0.5, random_state=0, stratify=y)

for name, X in [("one feature", X_single), ("eight correlated copies", X_redundant)]:
    model = GaussianNB().fit(X[train], y[train])
    probability = model.predict_proba(X[test])[:, 1]
    prediction = probability >= 0.5
    print(
        f"{name:24s}",
        f"accuracy={accuracy_score(y[test], prediction):.3f}",
        f"log-loss={log_loss(y[test], probability):.3f}",
        f"Brier={brier_score_loss(y[test], probability):.3f}",
        f"extreme={(np.abs(probability - 0.5) > 0.49).mean():.3f}",
    )
```

</details>

The model may preserve a reasonable decision boundary while its log loss and Brier score deteriorate. This is why naive Bayes should be evaluated with probability-sensitive metrics and a reliability diagram when probabilities, thresholds, or expected costs will be used.

Naive Bayes remains attractive for high-dimensional sparse baselines, streaming count updates, and situations where latency or interpretability is more important than extracting every last point of predictive accuracy. Its assumptions should be read as a computational design choice whose consequences can be tested, not as a claim that real features never interact.


### **Gaussian Discriminant Analysis**

Gaussian discriminant analysis replaces the feature-by-feature independence of Gaussian naive Bayes with a multivariate Gaussian model for each class. Correlations are represented through covariance matrices, so the model can describe tilted elliptical clouds rather than axis-aligned ones.

For class $c$ with prior $\pi_c$, mean $\mu_c$, and covariance $\Sigma_c$,

$$
p(x\mid y=c)
=\frac{1}{(2\pi)^{d/2}|\Sigma_c|^{1/2}}
\exp\left[-\frac12(x-\mu_c)^T\Sigma_c^{-1}(x-\mu_c)\right].
$$

The quadratic form is the squared **Mahalanobis distance**. Unlike Euclidean distance, it measures displacement relative to the class covariance: movement along a high-variance direction is less surprising than equal movement along a tightly concentrated direction. The determinant $|\Sigma_c|$ adjusts for the volume occupied by the class distribution.

Taking logs and dropping terms shared across classes gives the discriminant score

$$
\delta_c(x)
=-\frac12\log|\Sigma_c|
-\frac12(x-\mu_c)^T\Sigma_c^{-1}(x-\mu_c)
+\log\pi_c.
$$

Prediction selects the class with the largest score. The distinction between LDA and QDA is entirely about how covariance is shared.

#### **Linear Discriminant Analysis**

LDA assumes a common covariance matrix,

$$
\Sigma_1=\cdots=\Sigma_C=\Sigma.
$$

When the common quadratic term $-\tfrac12x^T\Sigma^{-1}x$ is removed from every class score, the remaining discriminant is linear in $x$:

$$
\delta_c^{LDA}(x)
=x^T\Sigma^{-1}\mu_c
-\frac12\mu_c^T\Sigma^{-1}\mu_c
+\log\pi_c.
$$

Pairwise equality of these scores defines a hyperplane. LDA therefore combines a generative Gaussian model with a linear decision boundary. The boundary can still be oblique because $\Sigma^{-1}$ accounts for correlated features.

For $C$ classes and $d$ features, LDA estimates $C$ means but only one $d\times d$ covariance matrix. This sharing reduces variance and is especially useful when each class has few observations. It also supports a supervised projection into at most $C-1$ discriminant dimensions, although classification and dimensionality reduction are distinct uses of the same fitted quantities.

<details>
<summary><strong>Python example: reproduce the LDA discriminant calculation</strong></summary>

```python
import numpy as np
from scipy.special import softmax
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis

rng = np.random.default_rng(4)
covariance = np.array([[1.2, 0.7], [0.7, 1.0]])
X0 = rng.multivariate_normal([-1.0, 0.0], covariance, size=120)
X1 = rng.multivariate_normal([1.0, 0.8], covariance, size=80)
X = np.vstack([X0, X1])
y = np.r_[np.zeros(len(X0), dtype=int), np.ones(len(X1), dtype=int)]

classes = np.unique(y)
priors = np.array([(y == c).mean() for c in classes])
means = np.vstack([X[y == c].mean(axis=0) for c in classes])

# Maximum-likelihood pooled within-class covariance: divide by total n.
scatter = sum((X[y == c] - means[c]).T @ (X[y == c] - means[c]) for c in classes)
pooled_covariance = scatter / len(X)
precision = np.linalg.inv(pooled_covariance)

X_new = np.array([[-0.5, 0.1], [0.4, 0.5], [1.5, 1.0]])
scores = np.column_stack([
    X_new @ precision @ means[c]
    - 0.5 * means[c] @ precision @ means[c]
    + np.log(priors[c])
    for c in classes
])
manual_probability = softmax(scores, axis=1)

sklearn_lda = LinearDiscriminantAnalysis(solver="lsqr").fit(X, y)
library_probability = sklearn_lda.predict_proba(X_new)

print("manual probabilities:\n", manual_probability.round(4))
print("scikit-learn probabilities:\n", library_probability.round(4))
print("maximum absolute difference:", np.max(np.abs(manual_probability - library_probability)))
```

</details>

#### **Quadratic Discriminant Analysis**

QDA estimates a separate covariance $\Sigma_c$ for every class. The class-specific $x^T\Sigma_c^{-1}x$ terms no longer cancel, so equal-score sets are quadratic surfaces. QDA can represent curved boundaries caused by classes with different orientations or spreads.

That flexibility is costly. Each full covariance matrix contains $d(d+1)/2$ unique entries. With $C$ classes, QDA estimates roughly $C$ times as many covariance parameters as LDA. If a class has no more than $d$ independent observations, its empirical covariance is singular; even before singularity, inversion can be numerically unstable. More data, dimensionality reduction, diagonal covariance, or regularization is then required.

<div class="diagram-scroll">

![Official scikit-learn comparison of LDA and QDA covariance ellipses and decision boundaries under shared and class-specific covariance structures.](assets/probabilistic-lda-qda.png){fig-alt="Official scikit-learn comparison of LDA and QDA covariance ellipses and decision boundaries under shared and class-specific covariance structures."}

</div>

*The top rows show cases where shared covariance is adequate; the bottom row shows QDA capturing a curved boundary when class covariances differ. Source: [scikit-learn LDA/QDA example](https://scikit-learn.org/stable/auto_examples/classification/plot_lda_qda.html).*

<details>
<summary><strong>Python example: compare LDA and QDA under shared and different covariances</strong></summary>

```python
import numpy as np
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis, QuadraticDiscriminantAnalysis
from sklearn.metrics import accuracy_score, log_loss

rng = np.random.default_rng(10)

def make_problem(n_per_class, covariance_0, covariance_1):
    X0 = rng.multivariate_normal([-0.8, 0.0], covariance_0, size=n_per_class)
    X1 = rng.multivariate_normal([0.8, 0.5], covariance_1, size=n_per_class)
    return np.vstack([X0, X1]), np.r_[np.zeros(n_per_class), np.ones(n_per_class)]

shared = np.array([[1.0, 0.65], [0.65, 1.0]])
different_0 = np.array([[1.4, 0.9], [0.9, 0.8]])
different_1 = np.array([[0.5, -0.35], [-0.35, 1.3]])

for setting, cov0, cov1 in [
    ("shared covariance", shared, shared),
    ("different covariances", different_0, different_1),
]:
    X_train, y_train = make_problem(250, cov0, cov1)
    X_test, y_test = make_problem(5_000, cov0, cov1)
    print(setting)
    for model in [LinearDiscriminantAnalysis(), QuadraticDiscriminantAnalysis()]:
        model.fit(X_train, y_train)
        probability = model.predict_proba(X_test)[:, 1]
        print(
            f"  {model.__class__.__name__:30s}",
            f"accuracy={accuracy_score(y_test, probability >= 0.5):.3f}",
            f"log-loss={log_loss(y_test, probability):.3f}",
        )
```

</details>

Covariance **shrinkage** moves an unstable empirical covariance toward a structured target, often a scaled identity matrix:

$$
\hat\Sigma_{shrink}=(1-\lambda)\hat\Sigma+\lambda\tau I,
\qquad 0\leq\lambda\leq1.
$$

Small $\lambda$ trusts the empirical correlations; large $\lambda$ suppresses them. This is not merely a numerical patch. It is a bias-variance decision about how much correlation structure the sample can support.

<details>
<summary><strong>Python example: stabilize high-dimensional LDA with covariance shrinkage</strong></summary>

```python
from sklearn.datasets import make_classification
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.model_selection import StratifiedKFold, cross_val_score

X, y = make_classification(
    n_samples=140,
    n_features=220,
    n_informative=18,
    n_redundant=12,
    class_sep=1.1,
    random_state=8,
)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=0)

models = {
    "empirical covariance": LinearDiscriminantAnalysis(solver="lsqr"),
    "automatic shrinkage": LinearDiscriminantAnalysis(solver="lsqr", shrinkage="auto"),
}
for name, model in models.items():
    scores = cross_val_score(model, X, y, cv=cv, scoring="neg_log_loss")
    print(f"{name:22s}: mean log-loss={-scores.mean():.3f} +/- {scores.std():.3f}")
```

</details>

| Model | Covariance assumption | Boundary | Statistical behavior |
|---|---|---|---|
| Gaussian NB | Diagonal $\Sigma_c$ per class | Usually quadratic | Lowest covariance cost; ignores correlations |
| LDA | One full shared $\Sigma$ | Linear | Strong pooling; lower variance |
| QDA | Full $\Sigma_c$ per class | Quadratic | Flexible; high variance and data demand |
| Shrinkage LDA/QDA | Covariance pulled toward a target | Linear or quadratic | Trades covariance bias for stability |

LDA and QDA should be selected by validation under realistic sample sizes, not by choosing the more flexible boundary by default. Their fitted probabilities should also be checked: even a visually plausible Gaussian boundary can be miscalibrated when class tails, outliers, or prevalence differ from the model.


### **Bayesian Parameter Learning**

Classical point estimation asks for one value of an unknown parameter. Bayesian parameter learning asks for a distribution that represents what remains plausible after combining prior information with observed evidence. The answer can be summarized by a mean or mode, but its spread and shape are part of the result rather than an optional error bar added afterward.

#### **Priors, Likelihoods, and Posteriors**

Bayes' rule for parameter $\theta$ and data $\mathcal D$ is

$$
p(\theta\mid\mathcal D)
=\frac{p(\mathcal D\mid\theta)p(\theta)}{p(\mathcal D)},
\qquad
p(\mathcal D)=\int p(\mathcal D\mid\theta)p(\theta)d\theta.
$$

- The **prior** $p(\theta)$ states uncertainty before the current data.
- The **likelihood** $p(\mathcal D\mid\theta)$ scores how well each parameter value explains the observations. As a function of $\theta$, it is not a probability distribution over $\theta$ until normalized with a prior.
- The **posterior** $p(\theta\mid\mathcal D)$ is the updated distribution.
- The **evidence** or marginal likelihood $p(\mathcal D)$ normalizes the posterior and measures how well a whole model, averaged over its prior, predicts the observed data.

For independent observations, likelihood terms multiply and log-likelihood terms add. As sample size grows, a regular identifiable model often becomes increasingly dominated by the data. With weak data, the prior and likelihood can both materially shape the posterior.

| Estimator | Optimization or integration | What is retained |
|---|---|---|
| Maximum likelihood | $\hat\theta_{MLE}=\arg\max_\theta p(\mathcal D\mid\theta)$ | One data-supported parameter value |
| Maximum a posteriori | $\hat\theta_{MAP}=\arg\max_\theta p(\mathcal D\mid\theta)p(\theta)$ | One prior-regularized parameter value |
| Full Bayesian inference | Compute or approximate $p(\theta\mid\mathcal D)$ | Distribution over plausible parameter values |

MAP estimation often resembles regularization: a zero-mean Gaussian prior produces an $L_2$ penalty, while a Laplace prior produces an $L_1$ penalty. The equivalence concerns the mode. Ridge or lasso coefficients alone do not provide a posterior distribution or posterior predictive uncertainty.

#### **Conjugate Priors**

A prior is **conjugate** to a likelihood when the posterior belongs to the same distribution family as the prior. Conjugacy turns integration and repeated updating into algebra on distribution parameters. It is especially useful for teaching, online updating, and components inside larger probabilistic models.

<div class="diagram-scroll">

![A Beta prior and Bernoulli observations update to a Beta posterior, from which the next-event probability is obtained.](assets/conjugate-update-flow.svg){fig-alt="A Beta prior and Bernoulli observations update to a Beta posterior, from which the next-event probability is obtained."}

</div>

For the Beta-Bernoulli model, the update is additive:

$$
\operatorname{Beta}(\alpha,\beta)
+(s\text{ successes},f\text{ failures})
\longrightarrow
\operatorname{Beta}(\alpha+s,\beta+f).
$$

The quantities $\alpha-1$ and $\beta-1$ are sometimes interpreted as prior pseudo-counts, but this analogy should not be pushed too far: the prior encodes a complete distribution, and parameterizations such as $\alpha+\beta$ control concentration.

<details>
<summary><strong>Python example: perform sequential Beta-Bernoulli updating</strong></summary>

```python
import numpy as np
from scipy.stats import beta

observations = np.array([1, 0, 1, 1, 1, 0, 1, 1, 0, 1])
alpha, beta_parameter = 2.0, 2.0

print("step  mean   95% credible interval")
for step, outcome in enumerate(observations, start=1):
    # A success increments alpha; a failure increments beta.
    alpha += outcome
    beta_parameter += 1 - outcome
    mean = alpha / (alpha + beta_parameter)
    interval = beta.ppf([0.025, 0.975], alpha, beta_parameter)
    print(f"{step:>4d}  {mean:.3f}  [{interval[0]:.3f}, {interval[1]:.3f}]")
```

</details>

For a $K$-class categorical outcome with class probabilities $\pi=(\pi_1,\ldots,\pi_K)$, the conjugate prior is Dirichlet:

$$
\pi\sim\operatorname{Dirichlet}(\alpha_1,\ldots,\alpha_K),
\qquad
\pi\mid\mathcal D\sim\operatorname{Dirichlet}(\alpha_1+n_1,\ldots,\alpha_K+n_K).
$$

Its posterior predictive probability for class $k$ is

$$
P(y_*=k\mid\mathcal D)=\frac{\alpha_k+n_k}{\sum_j(\alpha_j+n_j)}.
$$

<details>
<summary><strong>Python example: Dirichlet-Categorical posterior prediction</strong></summary>

```python
import numpy as np

classes = np.array(["basic", "plus", "premium"])
prior = np.array([1.0, 1.0, 1.0])
observed_counts = np.array([8, 3, 1])
posterior = prior + observed_counts
predictive = posterior / posterior.sum()

rng = np.random.default_rng(5)
probability_draws = rng.dirichlet(posterior, size=50_000)
intervals = np.quantile(probability_draws, [0.025, 0.975], axis=0)

for index, name in enumerate(classes):
    print(
        f"{name:7s}: predictive={predictive[index]:.3f}, "
        f"95% interval for class probability="
        f"[{intervals[0, index]:.3f}, {intervals[1, index]:.3f}]"
    )
```

</details>

Conjugate models include Beta-Binomial, Dirichlet-Multinomial, Gamma-Poisson, and Normal-Normal pairs. Their formulas are useful, but conjugacy should follow the scientific measurement model rather than dictate it. A convenient Gaussian likelihood is not defensible for strongly censored, bounded, or heavy-tailed observations merely because it yields a closed form.

#### **Posterior Predictive Distributions**

The posterior answers a parameter question; most deployed systems need an outcome question. The posterior predictive distribution is

$$
p(\tilde y\mid\mathcal D)
=\int p(\tilde y\mid\theta)p(\theta\mid\mathcal D)d\theta.
$$

For a Normal likelihood with known observation variance $\sigma^2$ and Normal prior

$$
y_i\mid\mu\sim\mathcal N(\mu,\sigma^2),
\qquad
\mu\sim\mathcal N(\mu_0,\tau_0^2),
$$

the posterior is Normal with

$$
\tau_n^2=\left(\frac{1}{\tau_0^2}+\frac{n}{\sigma^2}\right)^{-1},
\qquad
\mu_n=\tau_n^2\left(\frac{\mu_0}{\tau_0^2}+\frac{n\bar y}{\sigma^2}\right).
$$

The predictive distribution for one future observation is

$$
\tilde y\mid\mathcal D\sim\mathcal N(\mu_n,\sigma^2+\tau_n^2).
$$

The variance has two terms. $\sigma^2$ is irreducible outcome variability under the model; $\tau_n^2$ is uncertainty about the mean. More data shrinks the second term but not the first.

<details>
<summary><strong>Python example: separate parameter and outcome uncertainty</strong></summary>

```python
import numpy as np
from scipy.stats import norm

rng = np.random.default_rng(14)
true_mean = 2.0
observation_sd = 1.5
prior_mean, prior_sd = 0.0, 2.0
all_data = rng.normal(true_mean, observation_sd, size=100)

for n in [1, 5, 20, 100]:
    sample = all_data[:n]
    posterior_variance = 1.0 / (
        1.0 / prior_sd**2 + n / observation_sd**2
    )
    posterior_mean = posterior_variance * (
        prior_mean / prior_sd**2 + sample.sum() / observation_sd**2
    )
    predictive_sd = np.sqrt(observation_sd**2 + posterior_variance)
    interval = norm.interval(0.95, loc=posterior_mean, scale=predictive_sd)
    print(
        f"n={n:3d}: posterior mean={posterior_mean:.3f}, "
        f"parameter sd={np.sqrt(posterior_variance):.3f}, "
        f"predictive sd={predictive_sd:.3f}, interval={np.round(interval, 3)}"
    )
```

</details>

A **prior predictive check** draws parameters from the prior and then simulated data from the likelihood. It asks whether the prior implies observations on a plausible scale before fitting. A **posterior predictive check** repeats the process with posterior draws and compares replicated data with meaningful summaries of the real data. Neither check proves a model true; both can reveal assumptions that are incompatible with the phenomenon.

Prior sensitivity should be tested when data are weak or the model is only partially identified. Refit under multiple scientifically plausible priors and compare posterior predictions and decisions, not only parameter means. If operational conclusions change dramatically, that dependence is part of the result and should be reported.


### **Bayesian Linear and Logistic Regression**

Bayesian regression takes familiar prediction functions and treats their coefficients as uncertain. Instead of returning only $\hat w$, it produces $p(w\mid\mathcal D)$ and averages predictions over that distribution. The prior regularizes weakly supported directions, while the posterior covariance records which coefficient combinations remain uncertain.

#### **Bayesian Linear Regression**

Consider the Gaussian linear model

$$
y=Xw+\varepsilon,
\qquad
\varepsilon\sim\mathcal N(0,\sigma^2I),
$$

with Gaussian prior

$$
w\sim\mathcal N(m_0,S_0).
$$

Because the likelihood and prior are conjugate, the posterior is Gaussian:

$$
S_N=\left(S_0^{-1}+\frac{1}{\sigma^2}X^TX\right)^{-1},
$$

$$
m_N=S_N\left(S_0^{-1}m_0+\frac{1}{\sigma^2}X^Ty\right).
$$

$S_0^{-1}$ is prior precision and $X^TX/\sigma^2$ is data precision. Posterior precision is their sum. Directions in coefficient space that are well measured by $X$ become concentrated; collinear or weakly observed directions remain broad and are influenced more strongly by the prior.

At a new row vector $x_*$,

$$
y_*\mid x_*,\mathcal D
\sim
\mathcal N\left(x_*^Tm_N,\;\sigma^2+x_*^TS_Nx_*\right).
$$

The first variance term is observation noise. The second is coefficient uncertainty projected into the new input. It tends to be larger for extrapolation points because their feature combinations were not tightly constrained by training data.

<details>
<summary><strong>Python example: closed-form Bayesian polynomial regression</strong></summary>

```python
import numpy as np
from scipy.stats import norm

rng = np.random.default_rng(2)
x = np.linspace(-1.0, 1.0, 18)
y = 0.4 + 1.2 * x - 0.8 * x**2 + rng.normal(scale=0.25, size=len(x))

def design(values):
    # Intercept, linear term, and quadratic term.
    values = np.asarray(values)
    return np.column_stack([np.ones_like(values), values, values**2])

X = design(x)
noise_sd = 0.25
prior_mean = np.zeros(X.shape[1])
prior_covariance = 4.0 * np.eye(X.shape[1])

prior_precision = np.linalg.inv(prior_covariance)
posterior_covariance = np.linalg.inv(
    prior_precision + X.T @ X / noise_sd**2
)
posterior_mean = posterior_covariance @ (
    prior_precision @ prior_mean + X.T @ y / noise_sd**2
)

x_test = np.array([-1.5, 0.0, 1.5])
X_test = design(x_test)
mean = X_test @ posterior_mean
parameter_variance = np.einsum("ij,jk,ik->i", X_test, posterior_covariance, X_test)
predictive_sd = np.sqrt(noise_sd**2 + parameter_variance)
lower, upper = norm.interval(0.95, loc=mean, scale=predictive_sd)

print("posterior coefficient mean:", posterior_mean.round(3))
for value, prediction, sd, lo, hi in zip(x_test, mean, predictive_sd, lower, upper):
    print(f"x={value:>4.1f}: mean={prediction:>6.3f}, sd={sd:.3f}, 95% PI=[{lo:.3f}, {hi:.3f}]")
```

</details>

The interval widens outside the training range even though the assumed noise variance is constant. This is epistemic uncertainty from extrapolating the polynomial coefficients. It is still conditional on the polynomial model being appropriate; a narrow interval cannot account for a missing change point or an omitted variable unless the model represents that possibility.

<div class="diagram-scroll">

![Official scikit-learn Bayesian ridge curve-fitting example showing predictive means and uncertainty bands under different hyperparameter initializations.](assets/bayesian-ridge-uncertainty.png){fig-alt="Official scikit-learn Bayesian ridge curve-fitting example showing predictive means and uncertainty bands under different hyperparameter initializations."}

</div>

*The predictive band reflects uncertainty under the fitted Bayesian ridge model, while the differing fits show that empirical-Bayes hyperparameter optimization can depend on initialization. Source: [scikit-learn Bayesian ridge example](https://scikit-learn.org/stable/auto_examples/linear_model/plot_bayesian_ridge_curvefit.html).*

`BayesianRidge` commonly estimates coefficient and noise precisions by iteratively maximizing marginal likelihood. This is often called **empirical Bayes** or type-II maximum likelihood: hyperparameters are estimated as points rather than fully integrated under hyperpriors. It is useful and computationally efficient, but its uncertainty output should not be described as including all possible hyperparameter uncertainty.

<details>
<summary><strong>Python example: obtain predictive standard deviations with BayesianRidge</strong></summary>

```python
import numpy as np
from sklearn.linear_model import BayesianRidge
from sklearn.preprocessing import PolynomialFeatures

rng = np.random.default_rng(9)
x_train = rng.uniform(0.0, 1.0, size=30)
y_train = np.sin(2 * np.pi * x_train) + rng.normal(scale=0.15, size=30)

features = PolynomialFeatures(degree=5, include_bias=True)
X_train = features.fit_transform(x_train[:, None])
x_test = np.array([0.0, 0.5, 1.0, 1.4])
X_test = features.transform(x_test[:, None])

model = BayesianRidge(fit_intercept=False, compute_score=True).fit(X_train, y_train)
mean, standard_deviation = model.predict(X_test, return_std=True)

for x_value, prediction, sd in zip(x_test, mean, standard_deviation):
    print(f"x={x_value:.1f}: predictive mean={prediction:.3f}, sd={sd:.3f}")
print("final log marginal likelihood estimate:", round(model.scores_[-1], 3))
```

</details>

#### **Bayesian Logistic Regression**

Binary logistic regression uses

$$
P(y_i=1\mid x_i,w)=\sigma(x_i^Tw),
\qquad
\sigma(z)=\frac{1}{1+e^{-z}}.
$$

With Gaussian prior $w\sim\mathcal N(0,\Lambda^{-1})$, the log posterior is, up to a constant,

$$
\log p(w\mid X,y)
=\sum_i\left[y_i x_i^Tw-\log(1+e^{x_i^Tw})\right]
-\frac12w^T\Lambda w.
$$

Its mode is equivalent to $L_2$-regularized logistic regression under a matching penalty. Unlike Gaussian linear regression, the Bernoulli-logistic likelihood is not conjugate to a Gaussian prior. The posterior is not exactly Gaussian and the predictive probability

$$
P(y_*=1\mid x_*,\mathcal D)
=\int\sigma(x_*^Tw)p(w\mid\mathcal D)dw
$$

has no elementary closed form. Laplace approximation, variational inference, expectation propagation, or Monte Carlo methods are needed.

A Laplace approximation first finds the MAP coefficient $\hat w$. It then approximates the negative log posterior by a quadratic around that mode. If

$$
H=X^TWX+\Lambda,
\qquad
W_{ii}=\hat p_i(1-\hat p_i),
$$

then

$$
p(w\mid\mathcal D)\approx\mathcal N(\hat w,H^{-1}).
$$

The inverse Hessian is a local covariance estimate. It is informative when the posterior is approximately unimodal and quadratic near its mass, but it cannot represent strong skewness, heavy tails, or multiple modes.

<details>
<summary><strong>Python example: Laplace-approximate Bayesian logistic regression</strong></summary>

```python
import numpy as np
from scipy.optimize import minimize
from scipy.special import expit

rng = np.random.default_rng(15)
x = rng.normal(size=90)
true_probability = expit(-0.3 + 1.4 * x)
y = rng.binomial(1, true_probability)
X = np.column_stack([np.ones_like(x), x])

# A weak prior on the intercept and a stronger regularizing prior on the slope.
prior_precision = np.diag([0.05, 0.5])

def objective(w):
    z = X @ w
    negative_log_likelihood = np.sum(np.logaddexp(0.0, z) - y * z)
    return negative_log_likelihood + 0.5 * w @ prior_precision @ w

def gradient(w):
    return X.T @ (expit(X @ w) - y) + prior_precision @ w

fit = minimize(objective, x0=np.zeros(2), jac=gradient, method="BFGS")
map_w = fit.x
fitted_probability = expit(X @ map_w)
weights = fitted_probability * (1.0 - fitted_probability)
hessian = X.T @ (weights[:, None] * X) + prior_precision
laplace_covariance = np.linalg.inv(hessian)

# Integrate predictions over the approximate posterior by Monte Carlo.
x_test = np.array([-2.0, 0.0, 2.0])
X_test = np.column_stack([np.ones_like(x_test), x_test])
rng = np.random.default_rng(16)
w_draws = rng.multivariate_normal(map_w, laplace_covariance, size=100_000)
integrated_probability = expit(X_test @ w_draws.T).mean(axis=1)
plugin_probability = expit(X_test @ map_w)

print("MAP coefficients:", map_w.round(3))
print("posterior standard deviations:", np.sqrt(np.diag(laplace_covariance)).round(3))
for value, plug_in, integrated in zip(x_test, plugin_probability, integrated_probability):
    print(f"x={value:>4.1f}: plug-in={plug_in:.3f}, posterior-integrated={integrated:.3f}")
```

</details>

Posterior integration often moves extreme plug-in probabilities toward 0.5 when coefficient uncertainty is substantial, but this is not a universal calibration guarantee. A misspecified likelihood, biased sample, or shifted deployment population can still produce confidently wrong probabilities.

Bayesian linear and logistic regression are valuable when coefficient uncertainty, small-sample regularization, and decision-sensitive prediction matter. They remain conditional models: causal interpretation still requires an appropriate design and identification argument, and posterior uncertainty does not repair confounding or leakage.


### **Gaussian Processes**

A Gaussian process (GP) is a probability distribution over functions. Instead of choosing a finite set of basis coefficients first, it specifies which function values should be similar through a mean function $m(x)$ and covariance kernel $k(x,x')$:

$$
f\sim\mathcal{GP}(m,k).
$$

This notation means that for any finite inputs $x_1,\ldots,x_n$,

$$
\begin{bmatrix}f(x_1)\\ \vdots\\ f(x_n)\end{bmatrix}
\sim
\mathcal N\left(
\begin{bmatrix}m(x_1)\\ \vdots\\ m(x_n)\end{bmatrix},
K
\right),
\qquad K_{ij}=k(x_i,x_j).
$$

The kernel is therefore a prior over function structure. The radial basis function kernel

$$
k_{RBF}(x,x')=\sigma_f^2\exp\left(-\frac{\lVert x-x'\rVert^2}{2\ell^2}\right)
$$

uses amplitude $\sigma_f^2$ and length scale $\ell$. Small $\ell$ allows rapid variation; large $\ell$ couples distant inputs and favors smoother changes. Matérn kernels control differentiability more explicitly, periodic kernels encode repetition, and sums or products combine assumptions. Kernel choice is model design, not a decorative similarity option.

<div class="diagram-scroll">

![Official scikit-learn illustration of RBF Gaussian-process function samples before and after conditioning on observations.](assets/gp-rbf-prior-posterior.png){fig-alt="Official scikit-learn illustration of RBF Gaussian-process function samples before and after conditioning on observations."}

</div>

*Before fitting, each curve is a plausible function under the kernel prior. Conditioning forces posterior functions to agree near observations while leaving more variation elsewhere. Source: [scikit-learn GP prior/posterior example](https://scikit-learn.org/stable/auto_examples/gaussian_process/plot_gpr_prior_posterior.html).*

#### **Gaussian Process Regression**

Assume observations

$$
y=f(X)+\varepsilon,
\qquad
\varepsilon\sim\mathcal N(0,\sigma_n^2I).
$$

Let $K=K(X,X)$, $K_*=K(X,X_*)$, and $K_{**}=K(X_*,X_*)$. With zero prior mean, conditioning a joint Gaussian gives

$$
E[f_*\mid X,y,X_*]
=K_*^T(K+\sigma_n^2I)^{-1}y,
$$

$$
\operatorname{Cov}(f_*\mid X,y,X_*)
=K_{**}-K_*^T(K+\sigma_n^2I)^{-1}K_*.
$$

The mean is a kernel-weighted interpolation of observations. The covariance is prior uncertainty minus the part explained by correlations with training points. To predict noisy future observations rather than the latent function, add $\sigma_n^2I$ to the posterior covariance.

Implementations should use a Cholesky factorization rather than explicitly forming the inverse. A small positive diagonal **jitter** may be added for numerical stability; known observation noise belongs in the statistical model and should not be confused with purely numerical jitter.

<details>
<summary><strong>Python example: compute a Gaussian-process posterior from kernel matrices</strong></summary>

```python
import numpy as np
from scipy.linalg import cho_solve, solve_triangular
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF

rng = np.random.default_rng(6)
X = np.linspace(0.0, 5.0, 12)[:, None]
y = np.sin(X[:, 0]) + rng.normal(scale=0.15, size=len(X))
X_test = np.linspace(-0.5, 5.5, 40)[:, None]
length_scale = 0.9
noise_sd = 0.15

def rbf(A, B, ell):
    squared_distance = (A[:, None, :] - B[None, :, :]) ** 2
    return np.exp(-squared_distance.sum(axis=2) / (2.0 * ell**2))

K = rbf(X, X, length_scale) + noise_sd**2 * np.eye(len(X))
K_cross = rbf(X, X_test, length_scale)
K_test = rbf(X_test, X_test, length_scale)

L = np.linalg.cholesky(K)
alpha = cho_solve((L, True), y)
manual_mean = K_cross.T @ alpha
v = solve_triangular(L, K_cross, lower=True)
manual_variance = np.clip(np.diag(K_test) - np.sum(v**2, axis=0), 0.0, None)

library = GaussianProcessRegressor(
    kernel=RBF(length_scale=length_scale, length_scale_bounds="fixed"),
    alpha=noise_sd**2,
    optimizer=None,
).fit(X, y)
library_mean, library_sd = library.predict(X_test, return_std=True)

print("max mean difference:", np.max(np.abs(manual_mean - library_mean)))
print("max latent SD difference:", np.max(np.abs(np.sqrt(manual_variance) - library_sd)))
print("edge/interior latent SD:", np.round(library_sd[[0, 20, -1]], 3))
```

</details>

#### **Kernels and Function-Space Priors**

Kernel hyperparameters can be assigned priors and integrated, but many GP libraries choose them by maximizing the log marginal likelihood

$$
\log p(y\mid X)
=-\frac12y^T K_y^{-1}y
-\frac12\log|K_y|
-\frac n2\log(2\pi),
\qquad K_y=K+\sigma_n^2I.
$$

The first term rewards data fit; the log-determinant term penalizes covariance structures that spread probability mass too broadly; the final term is constant for fixed $n$. This automatic Occam effect is useful, but the objective can have local optima and remains conditional on the kernel family. Multiple optimizer restarts and input scaling are often necessary.

<details>
<summary><strong>Python example: inspect how RBF length scale changes evidence</strong></summary>

```python
import numpy as np
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF

rng = np.random.default_rng(11)
X = np.linspace(0.0, 4.0, 24)[:, None]
y = np.sin(2.2 * X[:, 0]) + rng.normal(scale=0.12, size=len(X))

for length_scale in [0.15, 0.5, 1.5, 5.0]:
    model = GaussianProcessRegressor(
        kernel=RBF(length_scale, length_scale_bounds="fixed"),
        alpha=0.12**2,
        optimizer=None,
    ).fit(X, y)
    adjacent_correlation = np.exp(
        -(X[1, 0] - X[0, 0]) ** 2 / (2.0 * length_scale**2)
    )
    print(
        f"length scale={length_scale:>4.2f}, "
        f"adjacent prior correlation={adjacent_correlation:.3f}, "
        f"log marginal likelihood={model.log_marginal_likelihood_value_:.3f}"
    )
```

</details>

An RBF GP assumes extreme smoothness and returns toward its prior away from data. That behavior may be sensible for spatial interpolation and unsafe for long-horizon extrapolation. Time series may require trend, periodic, and changing-noise components; spatial data may need anisotropic length scales; structured objects require kernels that respect their representation.

#### **Gaussian Process Classification**

For classification, a GP prior is placed on a latent function and transformed through a link:

$$
f\sim\mathcal{GP}(0,k),
\qquad
P(y_i=1\mid f_i)=\sigma(f_i)
$$

or a probit alternative. The Bernoulli likelihood breaks Gaussian conjugacy, so the posterior over latent values is not Gaussian. Laplace approximation, expectation propagation, variational inference, or sampling is required before predictive probabilities can be formed.

<div class="diagram-scroll">

![Official scikit-learn Gaussian-process classification probability surfaces for isotropic and feature-specific RBF length scales on Iris data.](assets/gp-classification-probabilities.png){fig-alt="Official scikit-learn Gaussian-process classification probability surfaces for isotropic and feature-specific RBF length scales on Iris data."}

</div>

*Color represents multiclass probability, not just a hard boundary. The anisotropic kernel learns a separate length scale for each feature. Source: [scikit-learn GPC Iris example](https://scikit-learn.org/stable/auto_examples/gaussian_process/plot_gpc_iris.html).*

<details>
<summary><strong>Python example: compare nonlinear GPC and linear logistic probabilities</strong></summary>

```python
import numpy as np
from sklearn.datasets import make_moons
from sklearn.gaussian_process import GaussianProcessClassifier
from sklearn.gaussian_process.kernels import RBF
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, log_loss
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

X, y = make_moons(n_samples=320, noise=0.24, random_state=3)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.45, stratify=y, random_state=4
)
scaler = StandardScaler().fit(X_train)
X_train = scaler.transform(X_train)
X_test = scaler.transform(X_test)

models = {
    "linear logistic": LogisticRegression(max_iter=2_000),
    "RBF Gaussian process": GaussianProcessClassifier(
        kernel=1.0 * RBF(length_scale=1.0),
        n_restarts_optimizer=1,
        random_state=0,
    ),
}
for name, model in models.items():
    model.fit(X_train, y_train)
    probability = model.predict_proba(X_test)[:, 1]
    print(
        f"{name:20s}: accuracy={accuracy_score(y_test, probability >= 0.5):.3f}, "
        f"log-loss={log_loss(y_test, probability):.3f}, "
        f"mean uncertainty={np.mean(1.0 - np.abs(2.0 * probability - 1.0)):.3f}"
    )
```

</details>

Exact GP regression requires $O(n^3)$ factorization time and $O(n^2)$ storage. Classification adds iterative approximate inference. This makes exact GPs compelling for small and medium datasets where structured uncertainty matters, but difficult at very large $n$. Inducing-point methods, structured kernel interpolation, random features, and local GPs trade exactness for scale.

| Model | Prior object | Exact posterior? | Main strength | Main limitation |
|---|---|---|---|---|
| Bayesian linear regression | Finite coefficient vector | Yes with Gaussian likelihood | Transparent global uncertainty | Fixed feature map |
| Bayesian logistic regression | Finite coefficient vector | No | Regularized probabilistic classification | Approximate integration required |
| GP regression | Function values | Yes with Gaussian likelihood | Flexible function uncertainty | Cubic exact scaling |
| GP classification | Latent function values | No | Nonlinear probability surfaces | Approximation plus cubic scaling |

Gaussian processes express a strong principle: uncertainty should grow where the data do not constrain the function. Whether that growth is realistic depends on the kernel and likelihood, so empirical coverage, calibration, and behavior under shift still have to be tested.


### **Approximate Bayesian Inference**

Exact Bayesian inference requires expectations under

$$
p(\theta\mid\mathcal D)
=\frac{p(\mathcal D,\theta)}{\int p(\mathcal D,\theta')d\theta'}.
$$

The denominator may be a high-dimensional integral, and even a known posterior density may not yield analytic predictive integrals. Nonconjugate likelihoods, hierarchical latent variables, mixture components, and neural networks quickly make exact computation unavailable. Approximate inference then becomes part of the model pipeline rather than an implementation detail.

Three major strategies answer different computational questions:

- **Laplace approximation:** Can the posterior be represented by local Gaussian curvature around a mode?
- **Variational inference:** Can inference be converted into optimization over a tractable distribution family?
- **Monte Carlo:** Can expectations be estimated from weighted or dependent random draws?

<div class="diagram-scroll">

![Laplace approximation, variational inference, and Monte Carlo methods provide different approximations to an intractable posterior.](assets/approximate-inference-map.svg){fig-alt="Laplace approximation, variational inference, and Monte Carlo methods provide different approximations to an intractable posterior."}

</div>

Approximation error is distinct from posterior uncertainty. A narrow variational distribution may reflect an under-dispersed approximation rather than strong evidence. Monte Carlo standard error is also distinct: it describes finite simulation accuracy for a posterior quantity, not how uncertain that quantity is under the model.

#### **Laplace Approximation**

Let

$$
h(\theta)=\log p(\mathcal D,\theta)
$$

be the log unnormalized posterior, and let $\hat\theta$ be its mode. A second-order Taylor expansion gives

$$
h(\theta)\approx h(\hat\theta)
-\frac12(\theta-\hat\theta)^T H(\theta-\hat\theta),
$$

where

$$
H=-\nabla^2 h(\hat\theta)
$$

is the negative Hessian at the mode. Exponentiating the quadratic yields

$$
p(\theta\mid\mathcal D)
\approx\mathcal N(\hat\theta,H^{-1}).
$$

The method is deterministic once the mode is found and reuses optimization curvature. It works well when posterior mass lies near one interior mode and the log density is approximately quadratic. It performs poorly for skewed, bounded, heavy-tailed, or multimodal posteriors, and it can fail when the Hessian is singular or not positive definite.

<details>
<summary><strong>Python example: compare a Laplace approximation with a numerical posterior</strong></summary>

```python
import numpy as np
from scipy.optimize import minimize_scalar

# Poisson observations with log-rate z and Gaussian prior z ~ N(0, 1.2^2).
counts = np.array([0, 0, 1])
prior_sd = 1.2

def log_posterior(z):
    # Terms constant in z are omitted.
    return counts.sum() * z - len(counts) * np.exp(z) - 0.5 * (z / prior_sd) ** 2

fit = minimize_scalar(lambda z: -log_posterior(z), bounds=(-7.0, 4.0), method="bounded")
mode = fit.x
negative_second_derivative = len(counts) * np.exp(mode) + 1.0 / prior_sd**2
laplace_sd = 1.0 / np.sqrt(negative_second_derivative)

# In one dimension, a dense grid gives a useful numerical reference.
grid = np.linspace(-7.0, 3.0, 60_000)
density = np.exp(np.array([log_posterior(z) for z in grid]) - log_posterior(mode))
density /= np.trapezoid(density, grid)
exact_mean = np.trapezoid(grid * density, grid)
exact_variance = np.trapezoid((grid - exact_mean) ** 2 * density, grid)
cdf = np.cumsum(density)
cdf /= cdf[-1]
exact_interval = np.interp([0.025, 0.975], cdf, grid)

print(f"Laplace on log-rate: mean={mode:.3f}, sd={laplace_sd:.3f}")
print(f"grid posterior:       mean={exact_mean:.3f}, sd={np.sqrt(exact_variance):.3f}")
print("grid 95% interval:", exact_interval.round(3))
print("Laplace 95% interval:", np.round([mode - 1.96 * laplace_sd, mode + 1.96 * laplace_sd], 3))
```

</details>

The approximation is made on the chosen parameterization. A Gaussian approximation to log-rate $z$ becomes log-normal for rate $\lambda=e^z$; approximating directly on $\lambda$ would give a different result and could put mass on impossible negative values. Parameterization is therefore part of Laplace accuracy.

#### **Variational Inference**

Variational inference (VI) chooses a tractable family $q_\phi(\theta)$ and fits its parameters $\phi$ to approximate the posterior. Directly minimizing

$$
\operatorname{KL}(q_\phi\Vert p)
=E_q\left[\log\frac{q_\phi(\theta)}{p(\theta\mid\mathcal D)}\right]
$$

appears to require the unknown evidence. Rearranging gives

$$
\log p(\mathcal D)
=\underbrace{E_q[\log p(\mathcal D,\theta)]-E_q[\log q_\phi(\theta)]}_{\operatorname{ELBO}(\phi)}
+\operatorname{KL}(q_\phi\Vert p).
$$

Because the evidence does not depend on $\phi$, maximizing the **evidence lower bound** (ELBO) minimizes the KL divergence. The first term rewards explaining data and prior structure; the entropy term discourages $q$ from collapsing without justification.

Mean-field VI factorizes $q(\theta)=\prod_jq_j(\theta_j)$. It is computationally convenient but cannot represent posterior correlations. The common reverse-KL direction strongly penalizes placing mass where the target has little density and weakly penalizes missing a separate mode. It can therefore select one mode and underestimate uncertainty.

<details>
<summary><strong>Python example: see reverse-KL Gaussian VI choose one mode</strong></summary>

```python
import numpy as np
from scipy.optimize import minimize
from scipy.special import logsumexp

# Symmetric bimodal target: 0.5 N(-2, 0.55^2) + 0.5 N(2, 0.55^2).
target_means = np.array([-2.0, 2.0])
target_sd = 0.55
nodes, weights = np.polynomial.hermite.hermgauss(80)
weights = weights / np.sqrt(np.pi)

def normal_logpdf(x, mean, sd):
    return -0.5 * ((x - mean) / sd) ** 2 - np.log(sd * np.sqrt(2.0 * np.pi))

def target_logpdf(x):
    components = np.vstack([
        np.log(0.5) + normal_logpdf(x, mean, target_sd)
        for mean in target_means
    ])
    return logsumexp(components, axis=0)

def reverse_kl(parameters):
    mean, log_sd = parameters
    sd = np.exp(log_sd)
    # Gauss-Hermite quadrature computes expectations under q=N(mean, sd^2).
    samples = mean + np.sqrt(2.0) * sd * nodes
    log_q = normal_logpdf(samples, mean, sd)
    return np.sum(weights * (log_q - target_logpdf(samples)))

for start in [[-1.5, -0.5], [1.5, -0.5]]:
    result = minimize(reverse_kl, start, method="BFGS")
    print(
        f"start mean={start[0]:>4.1f} -> q mean={result.x[0]:>6.3f}, "
        f"q sd={np.exp(result.x[1]):.3f}, KL={result.fun:.3f}"
    )

target_sd_total = np.sqrt(target_sd**2 + target_means[0] ** 2)
print(f"target mean=0.000, target marginal sd={target_sd_total:.3f}")
```

</details>

The optimized Gaussian covers one mode rather than averaging across the low-density valley. A richer mixture variational family could represent both modes. More generally, normalizing flows, structured covariance, and hierarchical variational families trade additional computation for expressiveness.

Large-data VI often uses minibatches and stochastic gradients. The reparameterization trick writes a Gaussian draw as $\theta=\mu+\sigma\odot\epsilon$, $\epsilon\sim\mathcal N(0,I)$, allowing gradients to pass through samples. ELBO convergence only establishes optimization within the selected family; it does not establish that the family is an accurate posterior approximation.

#### **Monte Carlo Methods**

Monte Carlo methods approximate expectations with random draws. For independent samples $\theta^{(s)}\sim p(\theta\mid\mathcal D)$,

$$
E[g(\theta)\mid\mathcal D]
\approx\frac1S\sum_{s=1}^{S}g(\theta^{(s)}).
$$

Direct posterior sampling is rarely available. **Importance sampling** draws from a proposal and reweights samples; it fails when a few weights dominate. **Markov chain Monte Carlo (MCMC)** constructs a dependent chain whose stationary distribution is the posterior. **Hamiltonian Monte Carlo (HMC)** uses gradients and simulated dynamics to make distant proposals with high acceptance in continuous spaces; the No-U-Turn Sampler adapts trajectory length.

Metropolis-Hastings proposes $\theta'\sim q(\theta'\mid\theta)$ and accepts with probability

$$
\alpha
=\min\left(1,
\frac{p(\mathcal D,\theta')q(\theta\mid\theta')}
{p(\mathcal D,\theta)q(\theta'\mid\theta)}
\right).
$$

With a symmetric random-walk proposal, the proposal terms cancel. Rejected proposals remain in the chain; deleting them would change the stationary distribution.

<details>
<summary><strong>Python example: sample a Beta-Binomial posterior with Metropolis-Hastings</strong></summary>

```python
import numpy as np
from scipy.special import expit
from scipy.stats import beta

rng = np.random.default_rng(22)
successes, failures = 14, 6
alpha_prior, beta_prior = 2.0, 2.0

def log_target_on_logit(z):
    theta = expit(z)
    # Beta posterior density plus the Jacobian d(theta)/dz=theta(1-theta).
    return (
        (alpha_prior + successes) * np.log(theta)
        + (beta_prior + failures) * np.log1p(-theta)
    )

n_steps = 16_000
burn_in = 3_000
proposal_sd = 0.75
z = 0.0
draws = np.empty(n_steps)
accepted = 0

for step in range(n_steps):
    proposal = z + rng.normal(scale=proposal_sd)
    log_ratio = log_target_on_logit(proposal) - log_target_on_logit(z)
    if np.log(rng.uniform()) < min(0.0, log_ratio):
        z = proposal
        accepted += 1
    draws[step] = expit(z)

posterior_draws = draws[burn_in:]

def teaching_ess(values, max_lag=1_000):
    # Initial-positive-pair estimate for illustration, not a replacement for Stan diagnostics.
    centered = values - values.mean()
    denominator = centered @ centered
    autocorrelation = [1.0]
    for lag in range(1, min(max_lag, len(values) - 1)):
        autocorrelation.append(centered[:-lag] @ centered[lag:] / denominator)
    paired_sum = 0.0
    for lag in range(1, len(autocorrelation) - 1, 2):
        pair = autocorrelation[lag] + autocorrelation[lag + 1]
        if pair <= 0:
            break
        paired_sum += pair
    return len(values) / (1.0 + 2.0 * paired_sum)

exact_a = alpha_prior + successes
exact_b = beta_prior + failures
print(f"acceptance rate: {accepted / n_steps:.3f}")
print(f"estimated ESS: {teaching_ess(posterior_draws):.0f} of {len(posterior_draws)} draws")
print(f"sample mean={posterior_draws.mean():.4f}, exact mean={exact_a / (exact_a + exact_b):.4f}")
print("sample interval:", np.quantile(posterior_draws, [0.025, 0.975]).round(4))
print("exact interval: ", beta.ppf([0.025, 0.975], exact_a, exact_b).round(4))
```

</details>

MCMC output is valid only after diagnosing the simulation. Multiple dispersed chains should reach the same typical region. Rank-normalized split $\hat R$ compares within- and between-chain behavior; bulk and tail effective sample size measure information after autocorrelation; Monte Carlo standard error quantifies simulation precision. HMC also requires checking divergent transitions, maximum tree depth, and energy diagnostics. The [Stan diagnostics documentation](https://mc-stan.org/rstan/reference/Rhat.html) recommends multiple chains and reports modern rank-normalized $\hat R$ together with bulk and tail ESS.

Thinning is generally useful only when storage is limiting. Keeping all post-warmup draws usually provides at least as much information as keeping every $k$th draw. More iterations do not repair a chain trapped in one mode, a biased variational family, or a misspecified model.

| Method | Representation | Computational style | Typical failure |
|---|---|---|---|
| Laplace | One local Gaussian | Mode finding and Hessian | Skewness, boundaries, multiple modes |
| Mean-field VI | Factorized tractable $q$ | ELBO optimization | Missing dependence and under-dispersion |
| Structured / flow VI | Richer transformed $q$ | Stochastic optimization | Harder optimization and remaining family bias |
| Importance sampling | Weighted proposal draws | Parallel sampling | Weight degeneracy |
| MCMC / HMC | Correlated posterior draws | Iterative simulation | Poor mixing, divergences, high cost |

Inference should be chosen according to posterior geometry, dataset size, latency, and the accuracy required for the final decision. On a new model, a smaller version that can be checked against numerical integration or high-quality MCMC is an effective way to validate a faster approximation.


### **Uncertainty, Calibration, and Model Comparison**

A posterior distribution is internally coherent under its model assumptions. It is not automatically calibrated to the external world. The likelihood may omit heavy tails, the prior may exclude plausible mechanisms, the training sample may be selected, and deployment inputs may be shifted. Uncertainty must therefore be evaluated as a predictive claim, not trusted because a model is labeled Bayesian.

#### **Aleatoric and Epistemic Uncertainty**

For parameter uncertainty $\theta\mid\mathcal D$, the law of total variance decomposes predictive variance as

$$
\operatorname{Var}(Y_*\mid x_*,\mathcal D)
=\underbrace{E_{\theta\mid\mathcal D}[\operatorname{Var}(Y_*\mid x_*,\theta)]}_{\text{aleatoric under the model}}
+\underbrace{\operatorname{Var}_{\theta\mid\mathcal D}[E(Y_*\mid x_*,\theta)]}_{\text{epistemic under the model}}.
$$

Aleatoric uncertainty describes outcome variability that remains for fixed parameters, such as overlapping classes or measurement noise. Epistemic uncertainty describes uncertainty about model quantities and can shrink with informative data. The decomposition is model-relative: uncertainty caused by an omitted variable may appear aleatoric until the variable is modeled, and a misspecified posterior may underestimate epistemic uncertainty entirely.

<div class="diagram-scroll">

![Predictive uncertainty contains aleatoric and epistemic components, both of which require calibration, coverage, shift, and decision checks.](assets/uncertainty-decomposition.svg){fig-alt="Predictive uncertainty contains aleatoric and epistemic components, both of which require calibration, coverage, shift, and decision checks."}

</div>

High epistemic uncertainty on out-of-domain inputs is desirable but not guaranteed. A GP may revert to its prior far from training points, while a neural network or linear model can extrapolate with extreme confidence. Explicit shift detection, representative validation, and safe fallback behavior remain necessary.

#### **Calibration and Proper Scoring Rules**

A binary probability predictor is calibrated when, among cases assigned probability $q$, the event occurs with frequency $q$:

$$
P(Y=1\mid\hat p(X)=q)=q.
$$

A reliability diagram bins predictions and compares mean predicted probability with observed event frequency. Points below the diagonal indicate overconfidence in the positive probability for that region; points above indicate underconfidence. Histograms are needed beside the curve because a bin with few cases is noisy and a model that only predicts near the base rate can look calibrated while having little discrimination.

<div class="diagram-scroll">

![Official scikit-learn reliability diagrams and probability histograms comparing logistic regression, naive Bayes, a margin model, and a random forest.](assets/probability-calibration-comparison.png){fig-alt="Official scikit-learn reliability diagrams and probability histograms comparing logistic regression, naive Bayes, a margin model, and a random forest."}

</div>

*The diagonal is ideal reliability; the lower histograms show where each model actually places probability mass. Source: [scikit-learn calibration comparison](https://scikit-learn.org/stable/auto_examples/calibration/plot_compare_calibration.html).*

Probability quality has several dimensions:

- **Calibration:** Do stated probabilities match conditional frequencies?
- **Discrimination:** Do high-risk cases receive higher scores than low-risk cases?
- **Sharpness:** Does the model make informative predictions away from the base rate, subject to calibration?
- **Decision value:** Do the probabilities produce low loss under relevant actions and costs?

Log loss and Brier score are strictly proper scoring rules: in expectation they encourage reporting the true probability. Binary log loss is

$$
-\frac1n\sum_i[y_i\log\hat p_i+(1-y_i)\log(1-\hat p_i)],
$$

while Brier score is

$$
\frac1n\sum_i(\hat p_i-y_i)^2.
$$

Both combine calibration and discrimination, so neither alone diagnoses reliability. Expected calibration error (ECE) summarizes binned gaps but depends strongly on binning and is not a proper scoring rule. It should accompany, not replace, a reliability diagram, log loss, Brier score, and uncertainty intervals for the calibration estimate.

<details>
<summary><strong>Python example: compare calibrated, overconfident, and underconfident probabilities</strong></summary>

```python
import numpy as np
from scipy.special import expit, logit
from sklearn.metrics import brier_score_loss, log_loss, roc_auc_score

rng = np.random.default_rng(31)
x = rng.normal(size=50_000)
true_probability = expit(1.3 * x - 0.25)
y = rng.binomial(1, true_probability)

predictions = {
    "calibrated": true_probability,
    "overconfident": expit(1.9 * logit(true_probability)),
    "underconfident": expit(0.55 * logit(true_probability)),
}

def expected_calibration_error(y_true, probability, n_bins=15):
    edges = np.linspace(0.0, 1.0, n_bins + 1)
    bin_id = np.clip(np.digitize(probability, edges) - 1, 0, n_bins - 1)
    error = 0.0
    for index in range(n_bins):
        mask = bin_id == index
        if mask.any():
            gap = abs(y_true[mask].mean() - probability[mask].mean())
            error += mask.mean() * gap
    return error

for name, probability in predictions.items():
    print(
        f"{name:14s}: log-loss={log_loss(y, probability):.4f}, "
        f"Brier={brier_score_loss(y, probability):.4f}, "
        f"AUC={roc_auc_score(y, probability):.4f}, "
        f"ECE={expected_calibration_error(y, probability):.4f}"
    )
```

</details>

All three transformations preserve ranking and therefore have nearly identical AUC, yet probability-sensitive metrics distinguish them. This is why a strong ranking score does not imply that decision thresholds or expected costs are trustworthy.

Post-hoc calibration can fit a sigmoid, isotonic mapping, or temperature parameter. The calibrator requires data not used to fit the base model; cross-fitting is useful when data are scarce. Calibration must be rechecked after prevalence or covariate shift, and a single global curve can hide subgroup miscalibration.

#### **Posterior Predictive Checks**

A posterior predictive check draws

$$
\theta^{(s)}\sim p(\theta\mid\mathcal D),
\qquad
y_{rep}^{(s)}\sim p(y\mid\theta^{(s)}),
$$

then compares a statistic $T(y)$ with $T(y_{rep}^{(s)})$. The statistic should target a scientifically meaningful failure mode: tail size, zero frequency, class imbalance, autocorrelation, maximum run length, or residual structure. A discrepancy shows that the fitted model cannot reproduce an aspect of the observations; a non-discrepancy does not prove adequacy for every purpose.

<details>
<summary><strong>Python example: detect heavy tails with a posterior predictive check</strong></summary>

```python
import numpy as np
from scipy.stats import invgamma, t

rng = np.random.default_rng(42)
observed = t.rvs(df=2, size=100, random_state=rng)  # Deliberately heavy-tailed data.
n = len(observed)

# Weak Normal-Inverse-Gamma prior for a Normal likelihood.
mu0, kappa0, alpha0, beta0 = 0.0, 0.01, 2.0, 2.0
sample_mean = observed.mean()
sum_squares = np.sum((observed - sample_mean) ** 2)
kappa_n = kappa0 + n
mu_n = (kappa0 * mu0 + n * sample_mean) / kappa_n
alpha_n = alpha0 + n / 2.0
beta_n = beta0 + 0.5 * sum_squares + (
    kappa0 * n * (sample_mean - mu0) ** 2 / (2.0 * kappa_n)
)

n_replications = 5_000
sigma2_draws = invgamma.rvs(alpha_n, scale=beta_n, size=n_replications, random_state=rng)
mu_draws = rng.normal(mu_n, np.sqrt(sigma2_draws / kappa_n))
replicated = rng.normal(
    loc=mu_draws[:, None],
    scale=np.sqrt(sigma2_draws)[:, None],
    size=(n_replications, n),
)

def standardized_maximum(values, axis=-1):
    mean = values.mean(axis=axis, keepdims=True)
    sd = values.std(axis=axis, ddof=1, keepdims=True)
    return np.max(np.abs((values - mean) / sd), axis=axis)

observed_statistic = standardized_maximum(observed)
replicated_statistic = standardized_maximum(replicated, axis=1)
posterior_predictive_p = np.mean(replicated_statistic >= observed_statistic)

print(f"observed standardized maximum: {observed_statistic:.3f}")
print(f"replicated median maximum: {np.median(replicated_statistic):.3f}")
print(f"posterior predictive tail probability: {posterior_predictive_p:.4f}")
```

</details>

A very small posterior predictive tail probability indicates that the Normal model rarely reproduces the observed tail statistic. A Student-$t$ likelihood, contamination component, or explicit outlier process may be more defensible. Removing the extreme observation without explaining the data-generating process would hide rather than solve the mismatch.

#### **Model Evidence, Predictive Comparison, and Averaging**

Bayesian model evidence integrates likelihood over the prior:

$$
p(\mathcal D\mid M)
=\int p(\mathcal D\mid\theta,M)p(\theta\mid M)d\theta.
$$

The Bayes factor $p(\mathcal D\mid M_1)/p(\mathcal D\mid M_2)$ updates prior odds between models. Evidence rewards fit while penalizing prior volume that predicts data unlike the observations. This makes it sensitive to prior scale: diffuse priors can strongly reduce evidence even when their posterior estimates look reasonable. Bayes factors should therefore use genuinely specified priors and sensitivity analysis, not arbitrary “noninformative” scales.

Predictive alternatives such as held-out log score, nested cross-validation, leave-one-out cross-validation, and WAIC focus on out-of-sample performance. They answer a different question from posterior odds over a closed list of models. No criterion eliminates the need to define the deployment population, refit preprocessing inside splits, and account for temporal or grouped dependence.

When multiple models remain plausible, **Bayesian model averaging** uses

$$
p(y_*\mid x_*,\mathcal D)
=\sum_m p(y_*\mid x_*,\mathcal D,M_m)p(M_m\mid\mathcal D).
$$

This propagates uncertainty about model identity instead of pretending the selected model was known in advance. Stacking of predictive distributions is a related approach that chooses weights for predictive performance rather than interpreting them as posterior model probabilities.

<details>
<summary><strong>Python example: combine model uncertainty with Bayesian model averaging</strong></summary>

```python
import numpy as np
from scipy.special import softmax

model_names = np.array(["linear", "tree", "Gaussian process"])
log_evidence = np.array([-42.0, -43.5, -47.0])
prior_model_probability = np.array([1 / 3, 1 / 3, 1 / 3])
event_probability = np.array([0.20, 0.55, 0.80])

log_posterior_weight = log_evidence + np.log(prior_model_probability)
posterior_model_probability = softmax(log_posterior_weight)
averaged_probability = posterior_model_probability @ event_probability

print("posterior model probabilities:")
for name, weight, probability in zip(
    model_names, posterior_model_probability, event_probability
):
    print(f"  {name:16s}: weight={weight:.3f}, event probability={probability:.2f}")
print(f"Bayesian model-averaged event probability: {averaged_probability:.3f}")

# Convert the averaged probability to an action using the deployment loss.
false_positive_cost, false_negative_cost = 1.0, 4.0
threshold = false_positive_cost / (false_positive_cost + false_negative_cost)
print(f"decision threshold={threshold:.3f}, action={int(averaged_probability > threshold)}")
```

</details>

The toy calculation illustrates mechanics, not a recommendation to use marginal likelihood casually. In real work, evidence values must come from a coherent model and prior specification, while stacking weights require leakage-safe predictive estimates.

#### **A Practical Probabilistic Workflow**

1. **Define the prediction and action separately.** State the target distribution, available actions, and consequential errors.
2. **Specify the observation model.** Choose likelihood support, noise structure, dependence, and missing-data assumptions that match how measurements arise.
3. **Choose and interrogate priors.** Simulate prior predictive observations and perform sensitivity analysis for weakly identified quantities.
4. **Fit with an inference method appropriate to the geometry.** Validate fast approximations on a smaller tractable version when possible.
5. **Diagnose computation.** Check optimizer stability, Hessian conditioning, ELBO behavior, or multi-chain MCMC diagnostics as appropriate.
6. **Check the fitted model.** Use posterior predictive checks, residual structure, and domain-specific discrepancy statistics.
7. **Evaluate predictions on untouched representative data.** Measure proper scores, calibration, interval coverage, subgroup behavior, and decision loss.
8. **Stress distribution shift.** Test prevalence changes, extrapolation, missingness, and fallback or abstention behavior.
9. **Report uncertainty with its conditions.** Distinguish outcome, parameter, model, approximation, and Monte Carlo uncertainty.

| Claim | Minimum supporting check | Common mistake |
|---|---|---|
| “The probability is reliable” | Reliability curve plus proper scores on representative data | Reporting AUC alone |
| “The interval is 95%” | Empirical coverage and width under relevant conditions | Treating a model-based interval as distribution-free |
| “MCMC converged” | Multiple chains, $\hat R$, bulk/tail ESS, trace and sampler diagnostics | Judging one smooth trace by eye |
| “VI is accurate” | Comparison with stronger inference or simulation on a tractable case | Treating ELBO convergence as posterior validation |
| “The Bayesian model knows what it does not know” | Shift and misspecification stress tests | Assuming posterior spread covers omitted mechanisms |
| “Model A is better” | Leakage-safe predictive or decision comparison with uncertainty | Selecting by training evidence or one noisy split |

Probabilistic modeling is most valuable when it makes uncertainty operational. The goal is not to attach intervals to every prediction, but to connect assumptions, evidence, predictive distributions, and losses in a chain that can be inspected and tested. When any link is weak, the correct response may be a broader model, more data, a different action, or an explicit refusal to automate the decision.
